# Fine-tuning do modelo médico (PubMedQA) no Google Colab

Este notebook reproduz o **Step 3** do projeto no Colab, usando GPU (T4 ou superior) para treinar o modelo com PEFT/QLoRA.

**Antes de rodar:**
1. Ative a GPU: Menu *Runtime → Change runtime type → Hardware accelerator: GPU*.
2. **Opção A (recomendado):** Clone o repositório do GitHub na célula **2** — basta rodar as células em ordem; o clone traz o código e, se você já tiver rodado a preparação antes, pode fazer upload só de `data/train.jsonl` e `data/dev.jsonl`, ou ter `data/ori_pqal.json` no repo e rodar `scripts/run_prepare_data.py` no Colab.
3. **Opção B:** Coloque a pasta **POSTECH-FIAP-FASE3** no Google Drive e use a célula de “Preparar ambiente” apontando para o Drive.
4. **Opção C:** Faça upload apenas de `train.jsonl` e `dev.jsonl` na célula de upload manual.

**Por que este modelo?** O modelo base é o **Qwen2.5-0.5B-Instruct**. Foi escolhido para caber na GPU do Colab (~15 GB) com quantização 4-bit (QLoRA), ser rápido de treinar e já seguir instruções. Veja a documentação em `config/finetune_defaults.py` no repositório.

## 1. Montar Google Drive e instalar dependências

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install -q "torch>=2.0" "transformers>=4.36" "datasets>=2.14" "peft>=0.7" "accelerate>=0.25" "trl>=0.7" "bitsandbytes"

## 2. Clonar o repositório do GitHub

Rode a célula abaixo para clonar o projeto. Depois disso, o código estará em `/content/POSTECH-FIAP-FASE3`. Se o repositório já tiver `data/train.jsonl` e `data/dev.jsonl`, pode seguir para o fine-tuning; se não, coloque `data/ori_pqal.json` na pasta `data/` (upload) e rode `!python scripts/run_prepare_data.py` na próxima célula, ou use a célula de upload manual de train/dev.

In [ ]:
!git clone https://github.com/thiagonespereira/POSTECH-FIAP-FASE3.git /content/POSTECH-FIAP-FASE3
%cd /content/POSTECH-FIAP-FASE3
print("Repositório em /content/POSTECH-FIAP-FASE3")

## 3. Preparar ambiente (se não usou o clone)

Se você **já clonou** na célula acima, pode pular esta célula. Caso contrário: use a pasta no Drive (ajuste `DRIVE_REPO`) ou a célula de upload para ter `train.jsonl` e `dev.jsonl`. Se clonou mas ainda não tem train/dev, coloque `ori_pqal.json` em `data/` e rode: `!python scripts/run_prepare_data.py`.

In [ ]:
# Se já clonou na célula anterior, o repo está em /content/POSTECH-FIAP-FASE3
# Senão, use o Drive ou /content/data (upload manual)
import os
import shutil
from pathlib import Path

REPO_DIR = Path("/content/POSTECH-FIAP-FASE3")
DRIVE_REPO = "/content/drive/MyDrive/POSTECH-FIAP-FASE3"

if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    DATA_DIR = REPO_DIR / "data"
    OUTPUT_DIR = REPO_DIR / "outputs" / "finetune_pqal"
    print("Usando repositório em /content/POSTECH-FIAP-FASE3 (clone ou já copiado)")
elif Path(DRIVE_REPO).exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    shutil.copytree(DRIVE_REPO, REPO_DIR)
    os.chdir(REPO_DIR)
    DATA_DIR = REPO_DIR / "data"
    OUTPUT_DIR = REPO_DIR / "outputs" / "finetune_pqal"
    print("Usando repositório copiado do Drive para /content/POSTECH-FIAP-FASE3")
else:
    os.makedirs("/content/data", exist_ok=True)
    DATA_DIR = Path("/content/data")
    OUTPUT_DIR = Path("/content/outputs/finetune_pqal")
    print("Aguardando upload de train.jsonl e dev.jsonl em /content/data (próxima célula)")
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

### (Opcional) Upload manual de train.jsonl e dev.jsonl

Use esta célula só se **não** estiver usando o repositório do Drive.

In [ ]:
# Só rode se data/ori_pqal.json existir e train.jsonl ainda não
from pathlib import Path
if (Path("/content/POSTECH-FIAP-FASE3/data/ori_pqal.json").exists() and 
    not Path("/content/POSTECH-FIAP-FASE3/data/train.jsonl").exists()):
    %cd /content/POSTECH-FIAP-FASE3
    !python scripts/run_prepare_data.py
else:
    print("Pulando: ou já existem train/dev, ou ori_pqal.json não está em data/")

In [ ]:
from google.colab import files
import os
from pathlib import Path
os.makedirs("/content/data", exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    if name.endswith(".jsonl"):
        os.rename(name, f"/content/data/{name}")
        print(f"Movido para /content/data/{name}")

if not (Path("/content/data/train.jsonl").exists() and Path("/content/data/dev.jsonl").exists()):
    print("AVISO: Envie train.jsonl e dev.jsonl (nomes exatos).")

## 4. Rodar o fine-tuning

Com o repositório em /content/POSTECH-FIAP-FASE3 (clone ou Drive), o script usa o código do repo. Certifique-se de que data/train.jsonl e data/dev.jsonl existem.

In [ ]:
import sys
from pathlib import Path

if Path("/content/POSTECH-FIAP-FASE3").exists():
    sys.path.insert(0, "/content/POSTECH-FIAP-FASE3")
    from config.finetune_defaults import (
        MODEL_NAME, NUM_EPOCHS, BATCH_SIZE, GRADIENT_ACCUMULATION_STEPS,
        LEARNING_RATE, MAX_SEQ_LENGTH, LORA_R, LORA_ALPHA, LORA_DROPOUT,
        LORA_TARGET_MODULES, SAVE_STRATEGY, SAVE_TOTAL_LIMIT,
        LOGGING_STEPS, WARMUP_RATIO, WEIGHT_DECAY,
    )
    from src.models.run_finetune import run_finetune

    run_finetune(
        Path("/content/POSTECH-FIAP-FASE3/data"),
        Path("/content/POSTECH-FIAP-FASE3/outputs/finetune_pqal"),
        model_name=MODEL_NAME,
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        max_seq_length=MAX_SEQ_LENGTH,
        use_4bit=True,
        lora_r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        lora_target_modules=LORA_TARGET_MODULES,
        save_strategy=SAVE_STRATEGY,
        save_total_limit=SAVE_TOTAL_LIMIT,
        logging_steps=LOGGING_STEPS,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
    )
else:
    print("Coloque o repositório em /content (via Drive) e rode a célula anterior, ou execute localmente: python scripts/train_finetune.py")

## 5. Salvar modelo no Drive (opcional)

Após o treino, copie a pasta de saída para o Drive para não perder ao desconectar.

In [ ]:
import shutil
from pathlib import Path

src = Path("/content/POSTECH-FIAP-FASE3/outputs/finetune_pqal")
dst = Path("/content/drive/MyDrive/POSTECH-FIAP-FASE3/outputs/finetune_pqal")

if src.exists() and Path("/content/drive/MyDrive").exists():
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"Modelo copiado para {dst}")
else:
    print("Ajuste os caminhos acima ou faça download manual da pasta outputs/finetune_pqal.")